In [1]:
import pandas as pd
import sys
from pathlib import Path
root_path = Path().resolve().parent.parent

sys.path.append(str(root_path))
from utils.cleaning import *
from utils.feature_engineering import *
from utils.transformer import *
from utils.encoding import *
from utils.scaling import *
from utils.feature_selection import *

from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv("../data/weatherAUS.csv")

### Data Cleaning

In [3]:
df.drop(columns=['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am', 'Date'], inplace=True)
# buang semua baris yang targetnya kosong sebelum di-preprocessing
df = df.dropna(subset=['RainTomorrow']).reset_index(drop=True)

In [4]:
df = handle_whitespace(df)

[handle_whitespace] Stripped whitespace on 6 columns


In [5]:
df = fix_dtypes(df, ["Date"])

[fix_dtypes] 'WindGustDir': object → category (16 unique values)
[fix_dtypes] 'WindDir9am': object → category (16 unique values)
[fix_dtypes] 'WindDir3pm': object → category (16 unique values)
[fix_dtypes] 'RainToday': object → category (2 unique values)
[fix_dtypes] 'RainTomorrow': object → category (2 unique values)


In [6]:
df = drop_duplicates(df)

[drop_duplicates] Removed 47 duplicate rows (142193 → 142146)


### Imputer outlier dan missing value

In [7]:
X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
X_train_numeric = ['Pressure9am', 'Pressure3pm', 'WindGustSpeed', 'Humidity3pm', 'Temp3pm', 'Rainfall', 'WindSpeed3pm', 'Humidity9am', 'Temp9am', 'WindSpeed9am', 'MinTemp', 'MaxTemp']
X_train_category = ['WindDir9am', 'WindGustDir', 'WindDir3pm', 'RainToday']

X_train, X_test = impute_median(X_train, X_test, X_train_numeric)
X_train, X_test = impute_mode(X_train, X_test, X_train_category)

[impute_median] 'Pressure9am': Imputed using train median = 1017.6000
[impute_median] 'Pressure3pm': Imputed using train median = 1015.2000
[impute_median] 'WindGustSpeed': Imputed using train median = 39.0000
[impute_median] 'Humidity3pm': Imputed using train median = 52.0000
[impute_median] 'Temp3pm': Imputed using train median = 21.1000
[impute_median] 'Rainfall': Imputed using train median = 0.0000
[impute_median] 'WindSpeed3pm': Imputed using train median = 19.0000
[impute_median] 'Humidity9am': Imputed using train median = 70.0000
[impute_median] 'Temp9am': Imputed using train median = 16.7000
[impute_median] 'WindSpeed9am': Imputed using train median = 13.0000
[impute_median] 'MinTemp': Imputed using train median = 12.0000
[impute_median] 'MaxTemp': Imputed using train median = 22.6000
[impute_mode] 'WindDir9am': Imputed using train mode = N
[impute_mode] 'WindGustDir': Imputed using train mode = W
[impute_mode] 'WindDir3pm': Imputed using train mode = SE
[impute_mode] 'RainToda

In [9]:
X_train, X_test = cap_outliers(X_train, X_test, X_train.select_dtypes(include=['float64', 'int64']).columns)

[cap_outliers] 'MinTemp': Clipped to IQR bounds [-6.2000, 30.6000]
[cap_outliers] 'MaxTemp': Clipped to IQR bounds [2.4500, 43.6500]
[cap_outliers] 'Rainfall': Clipped to IQR bounds [-0.9000, 1.5000]
[cap_outliers] 'WindGustSpeed': Clipped to IQR bounds [8.5000, 68.5000]
[cap_outliers] 'WindSpeed9am': Clipped to IQR bounds [-11.0000, 37.0000]
[cap_outliers] 'WindSpeed3pm': Clipped to IQR bounds [-3.5000, 40.5000]
[cap_outliers] 'Humidity9am': Clipped to IQR bounds [18.0000, 122.0000]
[cap_outliers] 'Humidity3pm': Clipped to IQR bounds [-5.0000, 107.0000]
[cap_outliers] 'Pressure9am': Clipped to IQR bounds [1001.0500, 1034.2500]
[cap_outliers] 'Pressure3pm': Clipped to IQR bounds [998.4000, 1032.0000]
[cap_outliers] 'Temp9am': Clipped to IQR bounds [-1.5000, 35.3000]
[cap_outliers] 'Temp3pm': Clipped to IQR bounds [2.3000, 40.7000]


### Feature Encoding

In [10]:
fitur_category = X_train.select_dtypes(exclude=[np.number]).columns
fitur_category

Index(['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday'], dtype='object')

In [11]:
for col in fitur_category:
    print(f"{len(X_train[col].unique())} unique values in '{col}'")

49 unique values in 'Location'
16 unique values in 'WindGustDir'
16 unique values in 'WindDir9am'
16 unique values in 'WindDir3pm'
2 unique values in 'RainToday'


In [12]:
X_train, X_test, _ = label_encode(X_train, X_test, ['RainToday'])

[label_encode] 'RainToday' successfully encoded.


In [13]:
le_target = LabelEncoder()

y_train = le_target.fit_transform(y_train.astype(str))
y_test = le_target.transform(y_test.astype(str))

In [14]:
X_train, X_test = target_encode(X_train, X_test, ['Location'], y_train)
X_train, X_test = one_hot_encode(X_train, X_test, ['WindGustDir', 'WindDir9am', 'WindDir3pm'])

[target_encode] 'Location' encoded safely using train targets.
[one_hot_encode] Encoded 3 columns.


### Feature Scaling

In [15]:
continuous_cols = [
    'MinTemp', 'MaxTemp', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 
    'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm'
]
X_train, X_test = log_transform(X_train, X_test, ['Rainfall'])
X_train, X_test, scaler = robust_scale(X_train, X_test, continuous_cols)

[log_transform] Applied log(x + 1.0000) to 'Rainfall'
[robust_scale] Robust-scaled 11 columns.
